In [1]:
import pandas as pd
import numpy as np


In [2]:
# ── 0. Load ────────────────────────────────────────────────────────────────────
# df is your installment-grain dataframe (one row per LoanID × InstallmentNumber)
# Assumes all columns from your schema are present.
df = pd.read_csv("xpd_2025_data.csv")

# ── 1. Sort — critical for all lagged features ─────────────────────────────────
df = df.sort_values(["LoanID", "InstallmentNumber"]).reset_index(drop=True)

# ── 2. Collected amount per installment (leakage-free) ─────────────────────────
# We don't know the split between partial/early/3p amounts from flags alone,
# so we use PaidOffThisInstall as the actual amount collected that installment.
# The flags tell us *how* it was collected; the amount is PaidOffThisInstall.

flag_cols = ["InstallCollected", "EarlyCollected", "ThirdPartyCollected", "PartialCollected"]

df["collected_amount"] = np.where(
    df[flag_cols].fillna(0).max(axis=1) == 1,
    df["PaidOffThisInstall"].fillna(0),
    0
)

df["collected_flag"] = (
    df[flag_cols]
    .fillna(0)
    .max(axis=1)
    .astype(int)
)

In [3]:
# ── 3. Cumulative payin ratio at installment k (leakage-free) ──────────────────
# Sum of collected amounts for installments 1..k, divided by OriginatedAmount.
# This is safe to use as a feature at installment k because it only looks back.

df["cum_collected"] = df.groupby("LoanID")["collected_amount"].cumsum()
df["cumulative_payin_ratio_at_k"] = df["cum_collected"] / df["OriginatedAmount"].replace(0, np.nan)

In [4]:

# ── 4. Stage B lagged features ─────────────────────────────────────────────────
# At installment k, Stage B can only see installments 1..k-1.
# We shift all installment-level signals by 1 within each loan group.

lag_cols = [
    "InstallmentStatus",
    "InstallStatusCode",
    "collected_flag",
    "collected_amount",
    "cumulative_payin_ratio_at_k",
    "InstallCollected",
    "EarlyCollected",
    "ThirdPartyCollected",
    "PartialCollected",
    "isInstallDefault",
]

for col in lag_cols:
    df[f"{col}_lag1"] = df.groupby("LoanID")[col].shift(1)

# Cumulative payin ratio as of k-1 — the key Stage B feature
df["cumulative_payin_ratio_lag1"] = df.groupby("LoanID")["cumulative_payin_ratio_at_k"].shift(1)
# At installment 1 there is no prior history — fill with 0
df["cumulative_payin_ratio_lag1"] = df["cumulative_payin_ratio_lag1"].fillna(0)

# Number of prior returns (failed payments) up to but not including installment k
df["is_return"] = (df["InstallmentStatus"].str.strip().str.upper() == "RETURN").astype(int)
df["num_prior_returns"] = df.groupby("LoanID")["is_return"].cumsum() - df["is_return"]
# ↑ subtract current row so it's strictly prior returns

# Days since origination at each installment (uses InstallDueDate as proxy)
df["InstallDueDate"] = pd.to_datetime(df["InstallDueDate"])
df["OriginationDate"] = pd.to_datetime(df["OriginationDate"])
df["days_since_origination"] = (df["InstallDueDate"] - df["OriginationDate"]).dt.days


In [5]:
# ── 5. Loan-level default labels for Stage A ───────────────────────────────────
# Identify the first installment where a default occurred per loan.

defaults = (
    df[df["isInstallDefault"] == 1]
    .groupby("LoanID")["InstallmentNumber"]
    .min()
    .rename("first_default_installment")
)

loan_labels = df.groupby("LoanID").agg(
    LoanPaidOff=("LoanPaidOff", "max"),
    isLoanDefault=("isLoanDefault", "max"),
    max_installment=("InstallmentNumber", "max"),
    InstallCollected_any=("InstallCollected", "max"),
    EarlyCollected_any=("EarlyCollected", "max"),
    ThirdPartyCollected_any=("ThirdPartyCollected", "max"),
    PartialCollected_any=("PartialCollected", "max"),
    OriginatedAmount=("OriginatedAmount", "first"),
    OriginationDate=("OriginationDate", "first"),
    CustType=("CustType", "first"),
    Frequency=("Frequency", "first"),
    NewlyScored=("NewlyScored", "first"),
    LoanStatus=("LoanStatus", "first"),
    AppYear=("AppYear", "first"),
    AppMonth=("AppMonth", "first"),
    AppWeek=("AppWeek", "first"),
    PortFolioID=("PortFolioID", "first"),
    Application_ID=("Application_ID", "first"),
).reset_index()

loan_labels = loan_labels.merge(defaults, on="LoanID", how="left")


In [6]:
# ── 6. FPD / SPD / TPD / Clean bucket labels ──────────────────────────────────

def assign_bucket(row):
    fdi = row["first_default_installment"]
    if pd.isna(fdi):
        # No default recorded
        if row["LoanPaidOff"] == 1:
            return "Clean"
        else:
            return "Unknown"  # Loan still open or status unclear — exclude from training
    fdi = int(fdi)
    if fdi == 1:
        return "FPD"
    elif fdi == 2:
        return "SPD"
    elif fdi == 3:
        return "TPD"
    else:
        return "Clean"  # Defaulted after installment 3 — treated as Clean for bucket purposes
        # You may want a separate "Late Default" bucket depending on your loan terms

loan_labels["bucket"] = loan_labels.apply(assign_bucket, axis=1)

print(loan_labels["bucket"].value_counts())
# Sanity check: Unknown should be a small % of closed loans


bucket
Clean      38401
FPD         9010
SPD         7036
TPD         4720
Unknown     4618
Name: count, dtype: int64


In [7]:
# ── 7. Payoff type ─────────────────────────────────────────────────────────────
# Only meaningful for Clean loans. Tells us *how* the loan was paid off.
# EarlyCollected on any installment before max = early payoff.
# ThirdPartyCollected on any installment = third party settlement.

early_payoff_loans = (
    df[df["EarlyCollected"] == 1]
    .groupby("LoanID")["InstallmentNumber"]
    .min()
    .rename("early_payoff_at_installment")
)

third_party_loans = (
    df[df["ThirdPartyCollected"] == 1]["LoanID"]
    .unique()
)

partial_close_loans = (
    df[(df["PartialCollected"] == 1) & (df["InstallmentNumber"] == df.groupby("LoanID")["InstallmentNumber"].transform("max"))]
    ["LoanID"].unique()
)

def assign_payoff_type(row):
    if row["bucket"] != "Clean":
        return None  # Payoff type only for Clean loans
    lid = row["LoanID"]
    if lid in third_party_loans:
        return "third_party"
    if lid in early_payoff_loans.index:
        return "early"
    if lid in partial_close_loans:
        return "partial_close"
    if row["InstallCollected_any"] == 1:
        return "full"
    return "unknown_clean"

loan_labels["payoff_type"] = loan_labels.apply(assign_payoff_type, axis=1)

print(loan_labels[loan_labels["bucket"] == "Clean"]["payoff_type"].value_counts())


payoff_type
early            20778
full             17285
third_party        259
unknown_clean       78
partial_close        1
Name: count, dtype: int64


In [8]:
# ── 8. Bucket-level payin ratios (training-time statistics, closed loans only) ─
# These are FIXED constants estimated from historical closed loans.
# They go into the Monte Carlo formula — never into model features.

total_collected_per_loan = (
    df.groupby("LoanID")["collected_amount"].sum().rename("total_collected")
)

loan_labels = loan_labels.merge(total_collected_per_loan, on="LoanID", how="left")
loan_labels["realized_payin_ratio"] = (
    loan_labels["total_collected"] / loan_labels["OriginatedAmount"].replace(0, np.nan)
)

closed_loans = loan_labels[loan_labels["bucket"] != "Unknown"]

bucket_payin_ratios = (
    closed_loans
    .groupby("bucket")["realized_payin_ratio"]
    .agg(["mean", "std", "count", "median"])
    .rename(columns={"mean": "payin_ratio_mean", "std": "payin_ratio_std",
                     "count": "n_loans", "median": "payin_ratio_median"})
    .round(4)
)

print("\nBucket-level payin ratios (historical closed loans):")
print(bucket_payin_ratios)



Bucket-level payin ratios (historical closed loans):
        payin_ratio_mean  payin_ratio_std  n_loans  payin_ratio_median
bucket                                                                
Clean             1.6895           0.8815    38401                 1.3
FPD               0.0886           0.3396     9010                 0.0
SPD               0.3701           0.3781     7036                 0.3
TPD               0.7007           0.4376     4720                 0.6


In [9]:
# ── 9. Build Stage A dataset ───────────────────────────────────────────────────
# One row per loan. Features known at origination only.
# Target: bucket label.

STAGE_A_FEATURES = [
    "CustType", "Frequency", "OriginatedAmount", "NewlyScored",
    "LoanStatus", "AppYear", "AppMonth", "AppWeek", "PortFolioID",
]

stage_a = loan_labels[loan_labels["bucket"] != "Unknown"][
    ["LoanID"] + STAGE_A_FEATURES + ["bucket", "payoff_type"]
].copy()

# Encode target as integer for XGBoost
bucket_map = {"FPD": 0, "SPD": 1, "TPD": 2, "Clean": 3}
stage_a["bucket_code"] = stage_a["bucket"].map(bucket_map)

print(f"\nStage A dataset: {len(stage_a)} loans")
print(stage_a["bucket"].value_counts(normalize=True).round(3))



Stage A dataset: 59167 loans
bucket
Clean    0.649
FPD      0.152
SPD      0.119
TPD      0.080
Name: proportion, dtype: float64


In [10]:
# ── 10. Build Stage B dataset ──────────────────────────────────────────────────
# One row per LoanID × InstallmentNumber.
# Target: collected_flag at installment k.
# Features: everything lagged to k-1.
# Exclude installment 1 (no prior history to lag from — optional depending on your approach).

STAGE_B_FEATURES = [
    "InstallmentNumber",
    "cumulative_payin_ratio_lag1",      # key predictor
    "InstallmentStatus_lag1",
    "InstallStatusCode_lag1",
    "collected_flag_lag1",
    "num_prior_returns",
    "days_since_origination",
    "Frequency",
    "CustType",
    "OriginatedAmount",
]

stage_b = df[df["InstallmentNumber"] > 1][  # drop installment 1 — no lag history
    ["LoanID"] + STAGE_B_FEATURES + ["collected_flag"]
].copy()

# Merge bucket label in so you can stratify or filter Stage B by bucket if needed
stage_b = stage_b.merge(loan_labels[["LoanID", "bucket"]], on="LoanID", how="left")

print(f"\nStage B dataset: {len(stage_b)} installment rows")
print(f"Collection rate by installment:")
print(stage_b.groupby("InstallmentNumber")["collected_flag"].mean().round(3).to_string())



Stage B dataset: 984799 installment rows
Collection rate by installment:
InstallmentNumber
2     0.528
3     0.398
4     0.319
5     0.256
6     0.209
7     0.176
8     0.147
9     0.123
10    0.108
11    0.094
12    0.080
13    0.069
14    0.058
15    0.048
16    0.041
17    0.036
18    0.030
19    0.025
20    0.019
21    0.163
22    0.151
23    0.125
24    0.123
25    0.069
26    0.008
27    0.010
28    0.011
29    0.000
30    0.000
31    0.000
32    0.000
33    0.000
34    0.000
35    0.000
36    0.000
37    0.000
38    0.000
39    0.000
40    0.000
41    0.000
42    0.000
43    0.000
44    0.000
45    0.000
46    0.000
47    0.000
48    0.000
49    0.000
50    0.000
51    0.000
56    0.000
57    0.000
58    0.000
59    0.000
60    0.000
61    0.000
62    0.000
63    0.000
64    0.000


In [11]:
#step 10 sanity checkess
print(type(stage_b))
print(stage_b.index)
print(stage_b.columns.tolist())


<class 'pandas.DataFrame'>
RangeIndex(start=0, stop=984799, step=1)
['LoanID', 'InstallmentNumber', 'cumulative_payin_ratio_lag1', 'InstallmentStatus_lag1', 'InstallStatusCode_lag1', 'collected_flag_lag1', 'num_prior_returns', 'days_since_origination', 'Frequency', 'CustType', 'OriginatedAmount', 'collected_flag', 'bucket']


In [12]:
# ── 11. Quick validation checks ────────────────────────────────────────────────

# Check: bucket rates should roughly sum to 1 across closed loans
bucket_dist = stage_a["bucket"].value_counts(normalize=True)
assert abs(bucket_dist.sum() - 1.0) < 0.001, "Bucket rates don't sum to 1"

# Check: no future leakage in Stage B features (all _lag1 columns should be NaN at inst 1)
inst1_lags = df[df["InstallmentNumber"] == 1]["cumulative_payin_ratio_lag1"]
# This column shouldn't exist yet at inst 1 — after fillna(0) it will be 0, which is correct
assert (df[df["InstallmentNumber"] == 1]["cumulative_payin_ratio_lag1"] == 0).all(), \
    "Lag feature at installment 1 is not 0 — check shift logic"

# Check: no TotalRealizedPayin in either feature set
for col in STAGE_A_FEATURES + STAGE_B_FEATURES:
    assert "TotalRealizedPayin" not in col, f"Leakage: {col}"
    assert "isLoanDefault" not in col, f"Leakage: {col}"
    assert "LoanPaidOff" not in col, f"Leakage: {col}"

print("\nAll validation checks passed.")

# ── Outputs ────────────────────────────────────────────────────────────────────
# stage_a            → feed into XGBoost Stage A model
# stage_b            → feed into LightGBM Stage B model
# bucket_payin_ratios → fixed constants for Monte Carlo formula
# loan_labels        → full loan-level summary with bucket, payoff_type, realized_payin_ratio


All validation checks passed.


## Modeling Phase

In [13]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, log_loss
import xgboost as xgb
import matplotlib.pyplot as plt


In [14]:
# ── 0. Prep Stage A dataset ────────────────────────────────────────────────────
# stage_a and loan_labels come from the data prep code

STAGE_A_FEATURES = [
    "CustType", "Frequency", "OriginatedAmount", "NewlyScored",
    "LoanStatus", "AppYear", "AppMonth", "AppWeek", "PortFolioID",
]

# Drop unknowns (still-open loans — no terminal event yet)
train = stage_a[stage_a["bucket"] != "Unknown"].copy()

# ── 1. Encode categoricals ─────────────────────────────────────────────────────
cat_cols = ["CustType", "Frequency", "LoanStatus"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    encoders[col] = le  # save for inference

X = train[STAGE_A_FEATURES].copy()
y = train["bucket_code"].copy()  # 0=FPD, 1=SPD, 2=TPD, 3=Clean



In [15]:
# ── 2. Class weights ───────────────────────────────────────────────────────────
# Clean will dominate. We don't want the model to just predict Clean always.
# XGBoost softmax doesn't take sample_weight directly per class, so we
# compute sample weights manually.

class_counts = y.value_counts()
total = len(y)
class_weights = {c: total / (len(class_counts) * n) for c, n in class_counts.items()}
sample_weights = y.map(class_weights).values

print("Class distribution:")
print(y.value_counts().rename({0:"FPD",1:"SPD",2:"TPD",3:"Clean"}))
print("\nClass weights:", {["FPD","SPD","TPD","Clean"][k]: round(v,3) 
                           for k,v in class_weights.items()})


Class distribution:
bucket_code
Clean    38401
FPD       9010
SPD       7036
TPD       4720
Name: count, dtype: int64

Class weights: {'Clean': 0.385, 'FPD': 1.642, 'SPD': 2.102, 'TPD': 3.134}


In [16]:
# ── 3. Cross-validated training ────────────────────────────────────────────────
# Stratified k-fold — important given class imbalance

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probs   = np.zeros((len(train), 4))  # out-of-fold predicted probabilities
oof_preds   = np.zeros(len(train), dtype=int)
fold_scores = []
models      = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    sw_tr = sample_weights[train_idx]

    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,   # important — prevents overfitting on small FPD/SPD/TPD classes
        eval_metric="mlogloss",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    probs = model.predict_proba(X_val)
    preds = np.argmax(probs, axis=1)

    oof_probs[val_idx] = probs
    oof_preds[val_idx] = preds

    score = log_loss(y_val, probs)
    fold_scores.append(score)
    models.append(model)
    print(f"Fold {fold+1} log-loss: {score:.4f}  |  best iteration: {model.best_iteration}")

print(f"\nMean CV log-loss: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")


Fold 1 log-loss: 0.8343  |  best iteration: 492
Fold 2 log-loss: 0.8460  |  best iteration: 329
Fold 3 log-loss: 0.8406  |  best iteration: 344
Fold 4 log-loss: 0.8573  |  best iteration: 431
Fold 5 log-loss: 0.8463  |  best iteration: 426

Mean CV log-loss: 0.8449 ± 0.0076


In [17]:
# ── 4. OOF evaluation ─────────────────────────────────────────────────────────
bucket_names = ["FPD", "SPD", "TPD", "Clean"]

print("\nOOF Classification Report:")
print(classification_report(y, oof_preds, target_names=bucket_names))

# Calibration check — are the predicted probabilities realistic?
# Average predicted probability per bucket should roughly match base rate
print("\nCalibration check (predicted mean prob vs actual base rate):")
for i, name in enumerate(bucket_names):
    pred_mean = oof_probs[:, i].mean()
    actual    = (y == i).mean()
    print(f"  {name}: predicted {pred_mean:.3f}  actual {actual:.3f}")



OOF Classification Report:
              precision    recall  f1-score   support

         FPD       0.41      0.54      0.47      9010
         SPD       0.29      0.31      0.30      7036
         TPD       0.19      0.42      0.26      4720
       Clean       0.97      0.74      0.84     38401

    accuracy                           0.64     59167
   macro avg       0.47      0.51      0.47     59167
weighted avg       0.74      0.64      0.67     59167


Calibration check (predicted mean prob vs actual base rate):
  FPD: predicted 0.173  actual 0.152
  SPD: predicted 0.174  actual 0.119
  TPD: predicted 0.180  actual 0.080
  Clean: predicted 0.473  actual 0.649


In [23]:
# ── 5. Revenue-weighted confusion matrix ──────────────────────────────────────
# Raw accuracy doesn't tell you what matters — a false Clean (predicted Clean,
# actually FPD) is far more damaging than a false FPD (predicted FPD, actually Clean).
# Weight each misclassification by the payin ratio difference.

payin_by_bucket = bucket_payin_ratios["payin_ratio_mean"].to_dict()  # from data prep

revenue_cost = np.zeros((4, 4))  # [actual, predicted]
for actual in range(4):
    for pred in range(4):
        if actual != pred:
            actual_name = bucket_names[actual]
            pred_name   = bucket_names[pred]
            # Cost = what you expected to get (predicted) minus what you actually get (actual)
            expected = payin_by_bucket.get(pred_name, 0)
            realized = payin_by_bucket.get(actual_name, 0)
            revenue_cost[actual, pred] = expected - realized  # positive = overestimated

print("\nRevenue cost matrix (predicted - actual payin ratio):")
print(pd.DataFrame(revenue_cost, index=bucket_names, columns=bucket_names).round(3))
# Weighted revenue error across OOF predictions
total_revenue_error = 0
for i in range(len(y)):
    actual = y.iloc[i]
    pred   = oof_preds[i]
    if actual != pred:
        total_revenue_error += revenue_cost[actual, pred]

avg_revenue_error_per_loan = total_revenue_error / len(y)
print(f"\nAvg revenue overestimation per loan (OOF): {avg_revenue_error_per_loan:.4f}")
print(f"This × OriginatedAmount gives $ impact per loan on average")




Revenue cost matrix (predicted - actual payin ratio):
         FPD    SPD    TPD  Clean
FPD    0.000  0.282  0.612  1.601
SPD   -0.282  0.000  0.331  1.319
TPD   -0.612 -0.331  0.000  0.989
Clean -1.601 -1.319 -0.989  0.000

Avg revenue overestimation per loan (OOF): -0.1800
This × OriginatedAmount gives $ impact per loan on average


In [24]:
# ── 6. Feature importance ──────────────────────────────────────────────────────
# Average importance across folds
importances = np.mean([m.feature_importances_ for m in models], axis=0)
feat_imp = pd.Series(importances, index=STAGE_A_FEATURES).sort_values(ascending=False)

print("\nFeature importances (avg across folds):")
print(feat_imp.round(4))


Feature importances (avg across folds):
LoanStatus          0.4429
AppYear             0.1781
CustType            0.1013
AppMonth            0.0636
AppWeek             0.0621
Frequency           0.0563
OriginatedAmount    0.0402
NewlyScored         0.0285
PortFolioID         0.0269
dtype: float32


In [25]:
# ── 7. Final model — retrain on full dataset ───────────────────────────────────
# Use the median best_iteration from CV folds to avoid overfitting

median_iterations = int(np.median([m.best_iteration for m in models]))
print(f"\nRetraining final model with {median_iterations} iterations")

final_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=median_iterations,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    random_state=42,
    n_jobs=-1,
)

final_model.fit(X, y, sample_weight=sample_weights)


Retraining final model with 426 iterations


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [30]:
# ── 8. Inference function ──────────────────────────────────────────────────────
# Takes a dataframe of new loans (at origination) and returns bucket probabilities

def predict_bucket_probs(new_loans_df):
    """
    Input:  dataframe with columns matching STAGE_A_FEATURES
    Output: dataframe with columns P_FPD, P_SPD, P_TPD, P_Clean
    """
    X_new = new_loans_df[STAGE_A_FEATURES].copy()
    
    for col in cat_cols:
        X_new[col] = encoders[col].transform(
            X_new[col].astype(float).astype(int).astype(str)
        )

    
    probs = final_model.predict_proba(X_new)
    
    return pd.DataFrame(probs, columns=["P_FPD", "P_SPD", "P_TPD", "P_Clean"],
                        index=new_loans_df.index)

# Quick test on training data
sample_probs = predict_bucket_probs(train.head(5))
print("\nSample bucket probabilities:")
print(sample_probs.round(3))


ValueError: y contains previously unseen labels: '1'

In [ ]:
# ── 9. Save ────────────────────────────────────────────────────────────────────
import joblib
joblib.dump(final_model, "stage_a_model.pkl")
joblib.dump(encoders,    "stage_a_encoders.pkl")
joblib.dump(feat_imp,    "stage_a_feature_importance.pkl")

print("\nStage A model saved.")